In [3]:
import numpy as np
import xarray as xr
import os
import sys
import subprocess
import threading


parent_dir = os.path.dirname(os.environ["GTE_DIR"].replace("Glaciation_time_estimator",""))
GTE_DIR=os.environ["GTE_DIR"]
sys.path.insert(0, parent_dir)
global CLAAS_FP
CLAAS_FP = os.environ["CLAAS_DIR"]
if CLAAS_FP == "":
    raise ValueError("CLAAS_DIR is not defined")

from Glaciation_time_estimator.Data_preprocessing.Resample_data import ProjectionTransformer

In [6]:
transformer = ProjectionTransformer()
aux_data = xr.load_dataset(os.path.join(
        CLAAS_FP, "np/CM_SAF_CLAAS3_L2_AUX.nc"), decode_times=False)
transformer.generate_lat_lon_prj(aux_data)


Bounds = [-79.74298161940857, 79.7396484770131, 26.044719326356088, 81.07635224475464]


In [ ]:


# Example new coordinate arrays (replace these with transformer.new_cord_lon and transformer.new_cord_lat)
new_cord_lon = transformer.new_cord_lon
new_cord_lat = transformer.new_cord_lat

# In your original code, the latitude array is flipped:
lat_for_grid = new_cord_lat[::-1]  # equivalent to np.flip(new_cord_lat)

# Determine grid attributes; for uniform spacing these must be constant differences.
xsize = len(new_cord_lon)
ysize = len(lat_for_grid)
xfirst = new_cord_lon[0]
xinc = new_cord_lon[1] - new_cord_lon[0] if xsize > 1 else 0  # ensure at least one point

yfirst = lat_for_grid[0]
yinc = lat_for_grid[1] - lat_for_grid[0] if ysize > 1 else 0   # this is expected to be negative if descending

# Write grid file
with open("grid.txt", "w") as f:
    f.write(f"gridtype = lonlat\n")
    f.write(f"xsize = {xsize}\n")
    f.write(f"ysize = {ysize}\n")
    f.write(f"xfirst = {xfirst}\n")
    f.write(f"xinc = {xinc}\n")
    f.write(f"yfirst = {yfirst}\n")
    f.write(f"yinc = {yinc}\n")


In [4]:
def dispatch(sem, argv, **kw):
    try:
        for args in argv[:-1]:
            subprocess.run(args, check=True,
                           stdout=subprocess.DEVNULL)
        for file in argv[-1]:
            os.remove(file)
    finally:
        sem.release()


def format_folder(folder_fp_ind, folder_fps_CTX, folder_fps_CPP, folder_resample_res_fps, do_resampling=False):
    sem = threading.Semaphore(8)   # pick a threshold here

    Ts = []
    cpp_fp_list = folder_fps_CPP[folder_fp_ind]
    ctx_fp_list = folder_fps_CTX[folder_fp_ind]
    reformated_fp_list= folder_resample_res_fps[folder_fp_ind]
    for filename_ind in range(len(cpp_fp_list)):
        cpp_fp = cpp_fp_list[filename_ind]
        ctx_fp = ctx_fp_list[filename_ind]
        reformated_output_fp = reformated_fp_list[filename_ind]
        merged_fp = reformated_output_fp.removesuffix(".nc")+"_merged.nc"
        # cdo -chname,cph,cph_resampled -setgrid,/wolke_scratch/dnikolo/Glaciation_time_estimator/Data_preprocessing/grid.txt -apply,-selname,cph [ CPPin20230101084500405SVMSGI1MD.nc ] test_1.nc
        argv = [["cdo", "merge", "-selname,ctt", ctx_fp, "-selname,cph", cpp_fp,merged_fp ],
                ["cdo", f"-chname,cph,cph_resampled","-chname,ctt,ctt_resampled", "-setgrid,/wolke_scratch/dnikolo/Glaciation_time_estimator/Data_preprocessing/grid.txt",
                    merged_fp, reformated_output_fp],
                [merged_fp]]
        sem.acquire()
        T = threading.Thread(target=dispatch, args=(sem, argv))
        T.start()
        Ts.append(T)

    for T in Ts:
        T.join()

In [5]:
folder_fps_CPP= [["/wolke_scratch/dnikolo/CLAAS_Data/np/2023/01/01/CPPin20230101003000405SVMSGI1MD.nc"],
                 ["/wolke_scratch/dnikolo/CLAAS_Data/np/2023/01/01/CPPin20230101003150405SVMSGI1MD.nc"]]
folder_fps_CTX= [["/wolke_scratch/dnikolo/CLAAS_Data/np/2023/01/01/CTXin20230101003000405SVMSGI1MD.nc"],
                 ["/wolke_scratch/dnikolo/CLAAS_Data/np/2023/01/01/CTXin20230101003150405SVMSGI1MD.nc"]]
folder_resample_res_fps = [["/wolke_scratch/dnikolo/dump/Data/np/test_1.nc"],
                           ["/wolke_scratch/dnikolo/dump/Data/np/test_2.nc"]]

In [6]:
format_folder(0,folder_fps_CTX,folder_fps_CPP,folder_resample_res_fps)